In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    Flatten,
    GlobalAveragePooling2D,
    Input
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

I0000 00:00:1785240185.654766  604240 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785240185.712099  604240 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785240187.214858  604240 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
val_df = pd.read_csv("Datasets/val.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 8)
(1502, 8)
(1503, 8)


In [4]:
#Encode Labels
encoder = LabelEncoder()

train_df["label"] = encoder.fit_transform(train_df["dx"])
val_df["label"] = encoder.transform(val_df["dx"])
test_df["label"] = encoder.transform(test_df["dx"])

print(encoder.classes_)

['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']


In [5]:
#Convert Labels to String

train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [6]:
#Constants
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5
NUM_CLASSES = 7

In [7]:
#Class_Weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{'0': np.float64(4.37305053025577), '1': np.float64(2.7817460317460316), '2': np.float64(1.3022478172023035), '3': np.float64(12.36331569664903), '4': np.float64(1.285530900421786), '5': np.float64(0.21338772031292808), '6': np.float64(10.115440115440116)}


In [8]:
#Data Augmentation (same for every model)
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [11]:
#Train Generator
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="image_path",

    y_col="dx",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 0 validated image filenames belonging to 0 classes.


/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/legacy/preprocessing/image.py:919: UserWarning: Found 7010 invalid image filename(s) in x_col="image_path". These filename(s) will be ignored.
  warnings.warn(


In [ ]:
#Validation Generator
val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False

)

In [10]:
#Test Generator
test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False

)

KeyError: 'path'

In [ ]:
#Display Sample Images
plt.figure(figsize=(10,6))

for i in range(6):

    plt.subplot(2,3,i+1)

    plt.imshow(images[i])

    plt.title(f"Class : {np.argmax(labels[i])}")

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
#Check One Batch
images, labels = next(train_generator)

print("Image Batch Shape :", images.shape)
print("Label Batch Shape :", labels.shape)

In [ ]:
#Build Deep CNN
deep_cnn = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),

    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

deep_cnn.summary()

In [ ]:
#Compile
deep_cnn.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
#Train
history_deep = deep_cnn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=EPOCHS,

    class_weight=class_weights

)

In [ ]:
#Evaluate
test_loss, test_acc = deep_cnn.evaluate(test_generator)

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss : {test_loss:.4f}")

In [ ]:
#Predictions
pred = deep_cnn.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

In [ ]:
#Metrics
accuracy = accuracy_score(
    true_labels,
    pred_labels
)

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


In [ ]:
#Save Results
results["Model"].append("Deep CNN")

results["Train Accuracy"].append(
    history_deep.history["accuracy"][-1]
)

results["Validation Accuracy"].append(
    history_deep.history["val_accuracy"][-1]
)

results["Test Accuracy"].append(accuracy)

results["Precision"].append(precision)

results["Recall"].append(recall)

results["F1 Score"].append(f1)

In [ ]:
#Save Model
deep_cnn.save("models/deep_cnn1.keras")

In [ ]:
#CNN + Batch Normalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)

cnn_bn = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),

    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

cnn_bn.summary()

In [ ]:
#Compile Model
cnn_bn.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
#Train Model
history_bn = cnn_bn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=EPOCHS,

    class_weight=class_weights

)

In [ ]:
#Evaluate
test_loss, test_acc = cnn_bn.evaluate(test_generator)

print("Test Accuracy :", round(test_acc,4))

print("Test Loss :", round(test_loss,4))

In [ ]:
#Metrics
accuracy = accuracy_score(true_labels, pred_labels)

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

In [ ]:
#SaveResults
results["Model"].append("CNN + Batch Normalization")

results["Train Accuracy"].append(
    history_bn.history["accuracy"][-1]
)

results["Validation Accuracy"].append(
    history_bn.history["val_accuracy"][-1]
)

results["Test Accuracy"].append(accuracy)

results["Precision"].append(precision)

results["Recall"].append(recall)

results["F1 Score"].append(f1)

In [ ]:
#Save Model
cnn_bn.save("models/cnn_bn.keras")

In [ ]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

mobilenet_model = Model(inputs, outputs)

mobilenet_model.summary()

In [ ]:
#Compile
mobilenet_model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
#Train
history_mobile = mobilenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=EPOCHS,

    class_weight=class_weights

)

In [ ]:
#Evaluate
test_loss, test_acc = mobilenet_model.evaluate(test_generator)

print("Test Accuracy :", round(test_acc,4))

print("Test Loss :", round(test_loss,4))

In [ ]:
#predictions
pred = mobilenet_model.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

In [ ]:
#Metrics
accuracy = accuracy_score(
    true_labels,
    pred_labels
)

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

In [ ]:
#Save Results
results["Model"].append("MobileNetV2")

results["Train Accuracy"].append(
    history_mobile.history["accuracy"][-1]
)

results["Validation Accuracy"].append(
    history_mobile.history["val_accuracy"][-1]
)

results["Test Accuracy"].append(accuracy)

results["Precision"].append(precision)

results["Recall"].append(recall)

results["F1 Score"].append(f1)

In [ ]:
mobilenet_model.save("models/mobilenetv2.keras")

In [ ]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

In [ ]:
#Compile
densenet_model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
#Train
history_dense = densenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=EPOCHS,

    class_weight=class_weights)



In [ ]:
#Evaluate
test_loss, test_acc = densenet_model.evaluate(test_generator)

print("Test Accuracy :", round(test_acc,4))

print("Test Loss :", round(test_loss,4))

In [ ]:
#Predictions
pred = densenet_model.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

In [ ]:
#Metrics
accuracy = accuracy_score(
    true_labels,
    pred_labels
)

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

In [ ]:
#save results
results["Model"].append("DenseNet121")

results["Train Accuracy"].append(
    history_dense.history["accuracy"][-1]
)

results["Validation Accuracy"].append(
    history_dense.history["val_accuracy"][-1]
)

results["Test Accuracy"].append(accuracy)

results["Precision"].append(precision)

results["Recall"].append(recall)

results["F1 Score"].append(f1)

In [ ]:
densenet_model.save("models/DenseNet121.keras")

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

results_df

In [ ]:
results_df.to_csv("models/model_comparison_phase4.csv", index=False)

In [ ]:
results1 = {
    "Model": [],
    "Train Accuracy": [],
    "Validation Accuracy": [],
    "Test Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1 Score": []
}

In [ ]:
#Basic Cnn
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

basic_cnn = Sequential([

    Conv2D(32, (3,3), activation="relu", padding="same",
           input_shape=(IMG_SIZE, IMG_SIZE,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(128,activation="relu"),

    Dense(NUM_CLASSES,activation="softmax")

])

basic_cnn.summary()

In [ ]:
basic_cnn.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
history_basic = basic_cnn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights

)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Predictions
pred = basic_cnn.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)
true_labels = test_generator.classes

# Metrics
train_acc = history_basic.history["accuracy"][-1]
val_acc = history_basic.history["val_accuracy"][-1]
test_acc = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, average="weighted")
recall = recall_score(true_labels, pred_labels, average="weighted")
f1 = f1_score(true_labels, pred_labels, average="weighted")

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:

results1["Model"].append("Basic_Cnn")
results1["Train Accuracy"].append(
    history_basic.history["accuracy"][-1]
)

results1["Validation Accuracy"].append(
    history_basic.history["val_accuracy"][-1]
)

results1["Test Accuracy"].append(test_acc)

results1["Precision"].append(precision)

results1["Recall"].append(recall)

results1["F1 Score"].append(f1)

In [ ]:
#Save Model
basic_cnn.save("models/basic_cnn1.keras")

In [ ]:
print(np.unique(pred_labels,return_counts=True))

In [ ]:
#Cnn_DropOut
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense,Dropout

cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_dropout.summary()

In [ ]:
cnn_dropout.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
#Train
history_dropout = cnn_dropout.fit(

    train_generator,

    validation_data=val_generator,

    epochs=EPOCHS,

    class_weight=class_weights

)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Predictions
pred = cnn_dropout.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)
true_labels = test_generator.classes

# Metrics
train_acc = history_dropout.history["accuracy"][-1]
val_acc = history_dropout.history["val_accuracy"][-1]
test_acc = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, average="weighted")
recall = recall_score(true_labels, pred_labels, average="weighted")
f1 = f1_score(true_labels, pred_labels, average="weighted")

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:

results1["Model"].append("CNN-DROPOUT")
results1["Train Accuracy"].append(
    history_dropout.history["accuracy"][-1]
)

results1["Validation Accuracy"].append(
    history_dropout.history["val_accuracy"][-1]
)

results1["Test Accuracy"].append(test_acc)

results1["Precision"].append(precision)

results1["Recall"].append(recall)

results1["F1 Score"].append(f1)

In [ ]:
#Save Model
cnn_dropout.save("models/cnn_dropout.keras")

In [ ]:
print(results1)

In [ ]:
# create new data generator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
train_datagen_com = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)
 
val_datagen_com = ImageDataGenerator(
    preprocessing_function=preprocess_input
)
 
test_datagen_com = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
train_generator_com = train_datagen_eff.flow_from_dataframe(
    dataframe=train_df,
    x_col="path",
    y_col="label",
    target_size=(128,128),
    batch_size=16,
    class_mode="categorical",
    shuffle=True
)
 
val_generator_com = val_datagen_eff.flow_from_dataframe(
    dataframe=val_df,
    x_col="path",
    y_col="label",
    target_size=(128,128),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)
 
test_generator_com = test_datagen_eff.flow_from_dataframe(
    dataframe=test_df,
    x_col="path",
    y_col="label",
    target_size=(128,128),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

# Base Model
base_model = EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(128,128,3)

)

base_model.trainable = False

# Build Model
efficientnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256, activation="relu"),

    Dropout(0.5),

    Dense(NUM_CLASSES, activation="softmax")

])

efficientnet.summary()

In [ ]:
efficientnet.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
history_efficientnet = efficientnet.fit(

    train_generator_eff,

    validation_data=val_generator_eff,

    epochs=5,

    class_weight=class_weights

)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Predictions
pred = efficientnet.predict(test_generator_eff)

pred_labels = np.argmax(pred, axis=1)
true_labels = test_generator.classes

# Metrics
train_acc = history_efficientnet.history["accuracy"][-1]
val_acc = history_efficientnet.history["val_accuracy"][-1]
test_acc = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, average="weighted")
recall = recall_score(true_labels, pred_labels, average="weighted")
f1 = f1_score(true_labels, pred_labels, average="weighted")

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:

results1["Model"].append("EfficientNetB0")
results1["Train Accuracy"].append(
    history_efficientnet.history["accuracy"][-1]
)

results1["Validation Accuracy"].append(
    history_efficientnet.history["val_accuracy"][-1]
)

results1["Test Accuracy"].append(test_acc)

results1["Precision"].append(precision)

results1["Recall"].append(recall)

results1["F1 Score"].append(f1)

In [ ]:
efficientnet.save("models/EfficientNetB0.keras")

In [ ]:
#ResNet50
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE,IMG_SIZE,3)

)

base_model.trainable = False

resnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

resnet.summary()

In [ ]:
resnet.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
history_resnet = resnet.fit(

    train_generator_com,

    validation_data=val_generator_com,

    epochs=5,

    class_weight=class_weights

)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Predictions
pred = resnet.predict(test_generator)

pred_labels = np.argmax(pred, axis=1)
true_labels = test_generator.classes

# Metrics
train_acc = history_resnet.history["accuracy"][-1]
val_acc = history_resnet.history["val_accuracy"][-1]
test_acc = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, average="weighted")
recall = recall_score(true_labels, pred_labels, average="weighted")
f1 = f1_score(true_labels, pred_labels, average="weighted")

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:

results1["Model"].append("ResNet50")
results1["Train Accuracy"].append(
    history_resnet.history["accuracy"][-1]
)

results1["Validation Accuracy"].append(
    history_resnet.history["val_accuracy"][-1]
)

results1["Test Accuracy"].append(test_acc)

results1["Precision"].append(precision)

results1["Recall"].append(recall)

results1["F1 Score"].append(f1)

In [ ]:
resnet.save("ResNet50.keras")


In [ ]:
print(results1)

In [ ]:
final_df = pd.read_csv("models/model_comparison_phase4.csv")

In [ ]:
results1_df = pd.DataFrame(results1)

In [ ]:
final_df = pd.concat(
    [final_df, results1_df],
    ignore_index=True
)

In [ ]:
final_df = final_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)


In [ ]:
final_df.to_csv(
    "final_model_comparison_phase4.csv",
    index=False
)

final_df